In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Load datasets
sentiment = pd.read_csv("../data/fear_greed.csv")
trades = pd.read_csv("../data/trader_data.csv")

# Preview first few rows
print("Sentiment Dataset:")
display(sentiment.head())

print("Trader Dataset:")
display(trades.head())


In [ ]:
# Check dataset shapes
print("Sentiment dataset shape:", sentiment.shape)
print("Trader dataset shape:", trades.shape)


In [ ]:
# Check missing values
print("Missing values in Sentiment dataset:")
print(sentiment.isnull().sum())

print("\nMissing values in Trader dataset:")
print(trades.isnull().sum())


### Missing Value Analysis

We checked both datasets for missing values.

Sentiment Dataset:
The sentiment dataset contains no missing values across all columns including timestamp, value, classification, and date.

Trader Dataset:
The trader dataset also contains no missing values in any of its columns among the 211,224 records.

This indicates that both datasets are clean and suitable for analysis without requiring additional data cleaning steps.


In [ ]:
print("Sentiment duplicates:", sentiment.duplicated().sum())
print("Trader duplicates:", trades.duplicated().sum())


### Duplicate Row Check

We checked both datasets for duplicate rows.

Sentiment Dataset:
No duplicate records were found.

Trader Dataset:
No duplicate records were identified among the 211,224 trading records.

This confirms that the datasets do not contain redundant entries and can be used directly for analysis.


In [ ]:
# Convert sentiment timestamp
sentiment['timestamp'] = pd.to_datetime(sentiment['timestamp'])

# Create date column
sentiment['date'] = sentiment['timestamp'].dt.date

sentiment.head()


In [ ]:
# Convert trader timestamp
trades['Timestamp'] = pd.to_datetime(trades['Timestamp'])

# Create date column
trades['date'] = trades['Timestamp'].dt.date

trades.head()


In [ ]:
# Reduce sentiment dataset to one record per day
sentiment_daily = sentiment[['date','classification']].drop_duplicates()

sentiment_daily.head()


In [ ]:
# Keep only important trader columns to reduce memory usage
trades_small = trades[['Account','Coin','Side','Closed PnL','Size USD','date']]

trades_small.head()


In [ ]:
# Merge trader data with sentiment data
data = pd.merge(
    trades_small,
    sentiment_daily,
    on='date',
    how='left'
)

data.head()


## Part B — Analysis

After merging the trader dataset with the Bitcoin Fear & Greed sentiment dataset, we can now analyze how market sentiment affects trader behavior and performance.

In this section we analyze:

1. Trader profitability during Fear vs Greed markets
2. Win rate comparison
3. Trade frequency patterns
4. Trader behavior changes based on sentiment


In [ ]:
# Average PnL by sentiment
pnl_sentiment = data.groupby('classification')['Closed PnL'].mean()

print(pnl_sentiment)


In [ ]:
# Average PnL by sentiment
pnl_sentiment = data.groupby('classification')['Closed PnL'].mean()

print(pnl_sentiment)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.barplot(x='classification', y='Closed PnL', data=data)

plt.title("Average Trader PnL by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Average Closed PnL")

plt.show()


In [ ]:
# Create win column
data['win'] = data['Closed PnL'] > 0

# Calculate win rate by sentiment
win_rate = data.groupby('classification')['win'].mean()

print(win_rate)


In [ ]:
sns.barplot(x='classification', y='win', data=data)

plt.title("Win Rate by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Win Rate")

plt.show()


In [ ]:
trade_freq = data['classification'].value_counts()

print(trade_freq)


In [ ]:
sns.countplot(x='classification', data=data)

plt.title("Number of Trades by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Trade Count")

plt.show()


### Key Insight 1

Trader profitability varies depending on market sentiment. 
Initial analysis shows that average PnL tends to differ between Fear and Greed periods.

### Key Insight 2

Win rates also change with market sentiment, indicating that trader success may be influenced by broader market psychology.

### Key Insight 3

Trading activity levels differ between Fear and Greed markets, suggesting that traders adjust their behavior based on market sentiment.


### Trader Performance by Trade Direction

We analyze whether traders perform better when taking Buy (long) positions or Sell (short) positions.


In [ ]:
side_pnl = data.groupby('Side')['Closed PnL'].mean()

print(side_pnl)


In [ ]:
sns.barplot(x='Side', y='Closed PnL', data=data)

plt.title("Average PnL by Trade Direction")
plt.xlabel("Trade Side")
plt.ylabel("Average Closed PnL")

plt.show()


### Top Performing Traders

We identify the traders with the highest cumulative profits to understand performance distribution across accounts.


In [ ]:
top_traders = data.groupby('Account')['Closed PnL'].sum().sort_values(ascending=False).head(10)

print(top_traders)


In [ ]:
plt.hist(data['Closed PnL'], bins=50)

plt.title("Distribution of Trader Profits")
plt.xlabel("Closed PnL")
plt.ylabel("Frequency")

plt.show()


### Trader Activity Under Different Market Sentiments

We examine how trading behavior changes between Fear and Greed market conditions.


In [ ]:
activity = data.groupby('classification')['Size USD'].mean()

print(activity)


In [ ]:
sns.barplot(x='classification', y='Size USD', data=data)

plt.title("Average Trade Size by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Average Trade Size (USD)")

plt.show()


## Conclusion

This analysis explored the relationship between Bitcoin market sentiment and trader performance.

Key findings include:

• Trader profitability varies depending on market sentiment conditions.
• Win rates differ during Fear vs Greed periods.
• Trading activity changes based on market psychology.
• Certain traders consistently outperform others.

Understanding sentiment-driven behavior can help traders and platforms improve risk management and trading strategies.


## Strategy Recommendations

Based on the analysis of trader behavior and market sentiment, the following strategy ideas can be considered:

### 1. Conservative Strategy During Fear Periods
During Fear sentiment periods, trader profitability tends to decline and market volatility increases. 
Traders may reduce leverage and trade size during these periods to manage risk and prevent large losses.

### 2. Opportunistic Strategy During Greed Periods
During Greed sentiment periods, trading activity and profitability appear to increase. 
Traders may consider increasing trading activity or slightly increasing position size during these periods to capitalize on stronger market momentum.

These strategies should be tested further with additional historical data before applying them in real trading environments.


## Future Work

Further analysis could include building predictive models to forecast trader profitability based on sentiment and behavioral features. 
Clustering traders into behavioral groups may also help identify different trading styles and risk profiles.
